In [ ]:
from skimage.io import imread
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from wsi.wsi import WSI
from wsi.utils import img_show
from wsi.nets.vision_transformer import vit_small
from torchvision import transforms as pth_transforms
import torchvision.models as tvmodel



In [ ]:
patch_size = 16
model = vit_small(patch_size=patch_size)
state_dict = torch.load("./model/base/DINO/20_checkpoint.pth")["teacher"]
state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}
state_dict = {k.replace("backbone.", ""): v for k, v in state_dict.items()}
model.load_state_dict(state_dict,strict=False)
for p in model.parameters():
    p.requires_grad = False



In [ ]:

def de_norm(img,mean,std):
    for i,(m,s) in enumerate(zip(mean,std)):
        img[:,:,i] = img[:,:,i]*s+m
    return img

def n_head_att_view(vit_model,img,patch_size=16,transform=None,imshow=True):
    transform = pth_transforms.Compose([
        pth_transforms.ToPILImage(),
        pth_transforms.ToTensor(),
        pth_transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ]) if transform is None else transform
    img = transform(img)
    w, h = img.shape[1] - img.shape[1] % patch_size, img.shape[2] - img.shape[2] % patch_size
    img = img[:, :w, :h].unsqueeze(0)
    w_featmap = img.shape[-2] // patch_size
    h_featmap = img.shape[-1] // patch_size

    attentions = vit_model.get_last_selfattention(img)
    nh = attentions.shape[1] # number of head
    attentions = attentions[0, :, 0, 1:].reshape(nh, -1)
    attentions = attentions.reshape(nh, w_featmap, h_featmap)
    attentions = nn.functional.interpolate(attentions.unsqueeze(0), scale_factor=patch_size, mode="nearest")[0].detach().cpu().numpy()    
    img = img.permute(0,2,3,1)[0].detach().cpu().numpy()
    img = de_norm(img,(0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    img_list = [img]+list(attentions)
    if imshow:
        plt.figure(figsize=(18,4))
        for i,m in enumerate(img_list):
            print(m.max(),m.min())
            plt.subplot(1,len(img_list),i+1)
            plt.axis("off")
            plt.imshow(m)
        plt.show()
    return attentions


wsi = WSI("../../CAMELYON16/training/normal/normal_002.tif")
_,rec,_ = wsi.get_tissue_patches(1,8,256,r=0.5)
img = wsi.read_region(rec[500],1,(256,256))
_ = n_head_att_view(model,img)
